# 12 — Macrotopic Classifier

**Pipeline step:** Section 3.3 of the paper — the classifier actually used to extend topic labels from the manually-labeled sample to the full dataset.

**Purpose.**
Trains a Random Forest classifier directly on the **macro-topic** label (a manual grouping of the ~60 BERTopic topics into broader categories — `macro_map` below), using the manually-labeled sample as training data and UMAP-reduced embeddings as features, then applies it to the full dataset so every video ends up with a `macrotopic_pred`.

**Input:**
- `data/yt_te_toxicity.csv` — Full dataset with Perspective API toxicity scores (from `src/perspective_mat.py`).
- `data/sample_yt_te_with_tox_top.csv` — Manually topic-labeled sample, merged with toxicity (same file flagged as an open gap in `19_Time_analysis.ipynb` — no notebook/script in this repo currently produces it under this exact name; likely a manual join of `sample_yt_te_cross_with_topics.csv` from `11_Merging_tables.ipynb` with the `perspective_*` columns).
- `classification_models/embeddings.npy`, `classification_models/embeddings_reduced.npy` — same open gap noted in the old `12_Interest_Classification.ipynb`: no notebook currently saves these files to this path.

**Output:**
- `data/macrotopics_pred.csv` — Full dataset with `macrotopic_pred` (predicted macro-topic) and `macrotopic_prob` for every video. **This is the file the results section of the paper is built on** — consumed by `27_Organizando_Resultados copy.ipynb`.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
import pickle
from sklearn.model_selection import KFold, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import numpy as np

`macro_map` — manual mapping from each of the ~60 BERTopic topic IDs to a broader macro-topic label (English names here; an earlier Portuguese-labeled version of this same mapping is used in the abandoned `22_Macrotopics.ipynb`/`22_Macrotopics copy.ipynb` drafts):

In [2]:
macro_map = {
    # POLÍTICA
    2: "American Politics",
    45: "American Politics",
    27: "American Politics",
    28: "American Politics",
    35: "American Politics",
    38: "American Politics",
    57: "American Politics",

    # POLÍTICA EXTERIOR
    5: "Foreign Policy",
    18: "Foreign Policy",
    20: "Foreign Policy",
    22: "Foreign Policy",
    26: "Foreign Policy",
    33: "Foreign Policy",
    37: "Foreign Policy",
    39: "Foreign Policy",
    47: "Foreign Policy",
    48: "Foreign Policy",
    56: "Foreign Policy",
    58: "Foreign Policy",
    23: "Foreign Policy",

    # CRIME, SEGURANÇA E JUSTIÇA
    24: "Crime, Security and Justice",
    51: "Crime, Security and Justice",
    59: "Crime, Security and Justice",
    31: "Crime, Security and Justice",
    34: "Crime, Security and Justice",

    # RELIGIÃO 
    0: "Religion",
    12: "Religion",

    # TECNOLOGIA
    60: "Technology",
    43: "Technology",
    9: "Technology",

    # SAÚDE
    8: "Health",
    15: "Health",
    
    # MEIO AMBIENTE
    10: "Environment",
    36: "Environment",
    42: "Environment",
    44: "Environment",

    # ENTRETENIMENTO E CULTURA POP
    1: "Entertainment",
    7: "Entertainment",
    11: "Entertainment",
    30: "Entertainment",
    40: "Entertainment",
    25: "Entertainment",
    41: "Entertainment",

    # ESTILO DE VIDA E HOBBIES
    16: "Hobbies",
    17: "Hobbies",
    21: "Hobbies",
    54: "Hobbies",
    29: "Hobbies",
    14: "Hobbies",

    # GUERRA
    4: "War",
    6: "War",
    32: "War",
    46: "War",
    50: "War",

    # OUTROS
    -1: "Others",
    3: "Others",
    49: "Others",
    55: "Others",
    53: "Others",
    13: "Others",
    19: "Others",
    52: "Others"
}


Loads the full dataset (with Perspective toxicity) and the manually topic-labeled training sample:

In [3]:
df = pd.read_csv("data/yt_te_toxicity.csv")
df_train = pd.read_csv("data/sample_yt_te_with_tox_top.csv")

print(len(df))
print(len(df_train))

673587
67357


Maps each sample video's fine-grained `topic` to its `macro_topic` via `macro_map`:

In [4]:
df_train["macro_topic"] = df_train["topic"].map(macro_map)
display(df_train)

,video_id,text,channel_title,published_at,view_count,like_count,comment_count,occurrences,toxicity,severe_toxicity,...,identity_attack,topic,interest,perspective_insult,perspective_severe_toxicity,perspective_obscene,perspective_toxicity,perspective_identity_attack,perspective_threat,macro_topic
0,QKXr7N9c6Yo,roman islamic antichrist sonia azam explore bi...,Armageddon News,2021-10-11T16:36:04Z,2249.0,107.0,75.0,"[{'id': 28786, 'folder': 'channel_2165208912',...",0.003288,0.000109,...,0.000284,0,1,0.378466,0.169603,0.003076,0.453968,0.460282,0.119229,Religion
1,qRFdVxvlXRY,kamala interview prove weak hope prove point w...,Real America's Voice,2024-10-08T14:57:04Z,498.0,89.0,10.0,"[{'id': 33565, 'folder': 'channel_1937328148',...",0.000926,0.000104,...,0.000136,2,1,0.039152,0.003357,0.001997,0.104362,0.014953,0.009774,American Politics
2,Y8KzbOkN_TM,money ttaakaa syllabus baire epbong welcome ep...,Your Bong Guy,2024-11-07T12:23:25Z,1198216.0,103644.0,5457.0,"[{'id': 3555926, 'folder': 'channel_1279841660...",0.005160,0.000092,...,0.000166,-1,0,0.016177,0.002613,0.009595,0.030978,0.006845,0.008078,Others
3,B5yS8olxa80,save israel jeremy gimpel live update frontlin...,thelandofisrael,2023-11-28T16:48:23Z,11835.0,1668.0,212.0,"[{'id': 37539, 'folder': 'channel_1592308283',...",0.002251,0.000098,...,0.000200,4,1,0.024124,0.008583,0.007347,0.130411,0.102216,0.038221,War
4,JnvOJN5yF8s,kingdom planet official trailer cinemas yang b...,20th Century Studios Indonesia,2024-02-12T01:24:59Z,211122.0,1928.0,374.0,"[{'id': 422797, 'folder': 'channel_1905119305'...",0.001937,0.000101,...,0.000176,-1,0,0.016861,0.003147,0.028147,0.067380,0.010951,0.019055,Others
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67352,-yqaSl75CWg,donald trump black conservative federation hon...,MAGNO NEWS,2024-02-24T03:36:18Z,132923.0,4173.0,694.0,"[{'id': 37202, 'folder': 'channel_1511714855',...",0.010595,0.000125,...,0.000335,2,1,0.085581,0.013608,0.002221,0.205721,0.165871,0.022854,American Politics
67353,B-9TmfJKIDw,revenge vseasy bgmi revange vvsvclutch vsclutc...,𝐒∆𝐈𝐘𝐀𝐍 • 127K views • 6 hours ago,2024-01-09T21:03:28Z,76.0,6.0,0.0,"[{'id': 202730, 'folder': 'channel_1215588509'...",0.586373,0.004986,...,0.014184,-1,0,0.025929,0.010834,0.011866,0.083208,0.008694,0.008738,Others
67354,5UfW-xNR7QE,stop celebrate year bible israel gaza jesus pa...,Aries Entertainment Podcast,2024-02-16T14:44:17Z,946556.0,NaN,3684.0,"[{'id': 561053, 'folder': 'channel_1905119305'...",0.126986,0.000889,...,0.019822,4,1,0.047350,0.016448,0.004756,0.241222,0.249708,0.031660,War
67355,LwAVckmcj9k,know gaza bible israel jesus palestine youtube...,Aries Entertainment Podcast,2024-03-02T13:50:44Z,4215776.0,NaN,5746.0,"[{'id': 132237, 'folder': 'channel_1457622198'...",0.119244,0.000962,...,0.018202,4,1,0.029714,0.011978,0.002999,0.154903,0.133773,0.025271,War


Aligns row order between the full dataset and the embeddings arrays (same approach as the old `12_Interest_Classification.ipynb`): builds `video_id → row index` from `df`, then maps each labeled sample video to its corresponding embedding row:

In [5]:
# Corrige alinhamento de índices
df_embeddings = df[['video_id']].copy()
df_embeddings['embedding_index'] = np.arange(len(df))
df_train_merged = df_train.merge(df_embeddings, on='video_id', how='left')

missing = df_train_merged['embedding_index'].isna().sum()
print(f"Linhas sem embedding correspondente: {missing}")

Linhas sem embedding correspondente: 0


Selects the labeled training rows (`train_idx`) and target `y` (`macro_topic`), and loads both the full and UMAP-reduced embeddings, subsetting them to the labeled sample for training:

In [6]:
train_idx = df_train_merged['embedding_index'].dropna().astype(int).values
y = df_train_merged["macro_topic"]

embeddings = np.load("classification_models/embeddings.npy")
embeddings_reduced = np.load("classification_models/embeddings_reduced.npy")

X = embeddings[train_idx]
X_reduced = embeddings_reduced[train_idx]

`run_model` — same grid-search helper as `models.py`/the old `12`: scales features, runs `GridSearchCV` with 5-fold stratified CV scored by F1 macro, and reports a per-fold classification report:

In [7]:
def run_model(name, model, param_grid, X, y, cv_splits=5):
    print(f"\n Rodando modelo: {name}")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    strat_kfold = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)

    clf = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=strat_kfold,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1
    )

    clf.fit(X_scaled, y)
    best_model = clf.best_estimator_
    print(f"Melhores hiperparâmetros encontrados: {clf.best_params_}")

    print(f"=== {name} ===\n")
    print(f"Melhores parâmetros: {clf.best_params_}\n\n")

    reports = []
    for fold, (train_idx, test_idx) in enumerate(strat_kfold.split(X_scaled, y), 1):
        y_pred = best_model.fit(X_scaled[train_idx], y[train_idx]).predict(X_scaled[test_idx])
        report = classification_report(y[test_idx], y_pred, digits=3, output_dict=True)
        reports.append(report)

        print(f"\n--- Fold {fold} ---\n")
        print(classification_report(y[test_idx], y_pred, digits=3))
        print("\n")

    avg_f1_macro = np.mean([r['macro avg']['f1-score'] for r in reports])
    std_f1_macro = np.std([r['macro avg']['f1-score'] for r in reports])
    print(f"\nMédia F1 Macro: {avg_f1_macro:.3f} ± {std_f1_macro:.3f}\n")

    print(f"✅ Finalizado: {name} | Média F1 Macro: {avg_f1_macro:.3f}")
    return best_model

Trains the Random Forest on the reduced embeddings and keeps `best_model`:

In [8]:
params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, None],
    "min_samples_split": [2, 5],
    'class_weight': [None, 'balanced']
}
     

best_model = run_model(
    name="Random Forest",
    model=RandomForestClassifier(class_weight="balanced", random_state=42),
    param_grid=params,
    X=X_reduced,
    y=y,
    cv_splits=5
)



 Rodando modelo: Random Forest
Fitting 5 folds for each of 24 candidates, totalling 120 fits


/home/uselection/venv/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Melhores hiperparâmetros encontrados: {'class_weight': 'balanced', 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
=== Random Forest ===

Melhores parâmetros: {'class_weight': 'balanced', 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}



--- Fold 1 ---

                             precision    recall  f1-score   support

          American Politics      0.773     0.822     0.797      1016
Crime, Security and Justice      0.659     0.605     0.631       185
              Entertainment      0.771     0.793     0.782      2193
                Environment      0.727     0.776     0.751       223
             Foreign Policy      0.785     0.753     0.768       833
                     Health      0.795     0.753     0.773       231
                    Hobbies      0.687     0.712     0.699       389
                     Others      0.763     0.739     0.751      5040
                   Religion      0.833     0.859     0.846      2154
                 Techn

Sanity check: sample rows whose `topic` didn't map to any `macro_topic` (i.e. missing from `macro_map`):

In [9]:
df_train_merged[df_train_merged["macro_topic"].isna()][["video_id", "topic"]]


,video_id,topic


Refits a `StandardScaler` on the full reduced-embeddings training set, to scale the whole dataset consistently before predicting:

In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_reduced)

**Applies the trained model to every video in the dataset**: scales the full reduced-embeddings array with the same scaler, then predicts both the macro-topic class and a probability

In [11]:
# embeddings reduzidos para TODOS os vídeos
all_embeddings_reduced = embeddings_reduced

# aplica o mesmo scaler usado no treino
all_embeddings_scaled = scaler.transform(all_embeddings_reduced)

pred_all = best_model.predict(all_embeddings_scaled)
prob_all = best_model.predict_proba(all_embeddings_scaled)[:, 1]  


Attaches the predictions to the full dataframe:

In [12]:
df['macrotopic_pred'] = pred_all
df['macrotopic_prob'] = prob_all

Quick look at the result:

In [13]:
display(df)

,video_id,text,channel_title,published_at,view_count,like_count,comment_count,occurrences,toxicity,severe_toxicity,...,insult,identity_attack,perspective_insult,perspective_severe_toxicity,perspective_obscene,perspective_toxicity,perspective_identity_attack,perspective_threat,macrotopic_pred,macrotopic_prob
0,jEKzQV5oajY,travel migrant scrap minute,Joe Marsh,2024-04-10T15:05:37Z,1057.0,152.0,32.0,"[{'id': 1416, 'folder': 'channel_1556142220', ...",0.000659,0.000117,...,0.000178,0.000138,0.206683,0.021936,0.138744,0.360951,0.367026,0.038566,Others,0.000000
1,xrGGce8cmx8,spiritual warfare charge commit unto thee timo...,WWURD,2023-10-18T14:39:18Z,65.0,6.0,1.0,"[{'id': 60799, 'folder': 'channel_1466271872',...",0.295000,0.001137,...,0.003023,0.006275,0.021224,0.005379,0.008414,0.123468,0.049762,0.029415,Religion,0.000000
2,uaozGpSc4nc,putin layer february tucker carlson stand onio...,The Mosaic Ark,2024-02-22T07:30:05Z,330.0,8.0,6.0,"[{'id': 40486, 'folder': 'channel_1235978663',...",0.001735,0.000102,...,0.000204,0.000156,0.132458,0.010376,0.003292,0.229804,0.062040,0.033386,Others,0.021935
3,A59ftbhsQUE,GALACTIC ALLIANCE MESSAGE NEWS REMAIN ALERT PR...,GALACTIC ALLIANCE,2024-10-27T04:30:04Z,806.0,136.0,18.0,"[{'id': 449843, 'folder': 'channel_1571505334'...",0.002710,0.000104,...,0.000239,0.000192,0.019419,0.005112,0.001093,0.075294,0.017666,0.022509,Religion,0.005000
4,AKU0RokegSo,angeles rain studio city evacuate mudslide atm...,FOX 11 Los Angeles,2024-02-06T01:45:29Z,60863.0,406.0,152.0,"[{'id': 104116, 'folder': 'channel_1438734111'...",0.001515,0.000100,...,0.000203,0.000143,0.027841,0.004921,0.002368,0.111507,0.019375,0.028207,Environment,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
673582,ABpufr8inCY,eruption kilauea volcano earthquake activity i...,TheEarthMaster,2024-09-17T19:15:25Z,10563.0,712.0,38.0,"[{'id': 30195, 'folder': 'channel_1177617529',...",0.003324,0.000098,...,0.000247,0.000152,0.012187,0.028856,0.001717,0.004477,0.002119,0.008324,Environment,0.000000
673583,5VWWCeGrr-w,eslteacher funny kidsvideo play easy makeup le...,Game fanny,2024-05-08T22:00:03Z,2.0,0.0,0.0,"[{'id': 1531863, 'folder': 'channel_1391678616...",0.004620,0.000084,...,0.000313,0.000175,0.023093,0.083999,0.005302,0.011395,0.207752,0.010628,Others,0.091652
673584,RxbPFQp0SHs,unbelievable ending watch play bgmi thank watc...,𝐒∆𝐈𝐘𝐀𝐍 • 127K views • 6 hours ago,2023-12-29T23:30:11Z,82.0,4.0,0.0,"[{'id': 197485, 'folder': 'channel_1215588509'...",0.023877,0.000158,...,0.001188,0.000347,0.017089,0.004711,0.006375,0.063423,0.010581,0.012907,Others,0.000000
673585,GEhEUy85PsY,meet balkan,Icariaball,2021-04-22T16:25:46Z,7617763.0,210152.0,14042.0,"[{'id': 7719, 'folder': 'channel_1753496417', ...",0.007485,0.000123,...,0.000321,0.000520,0.025156,0.094817,0.008087,0.016662,0.045911,0.020955,Others,0.026730


Saves the full dataset with macro-topic predictions to `data/macrotopics_pred.csv` — the file the rest of the results pipeline (from here on) is built on:

In [14]:
df.to_csv("data/macrotopics_pred.csv", index=False)